# CH09 — Ejercicios: ¿Se puede construir un *stack* y una *queue* con una cola de prioridad?

Ejercicios del capítulo 9 (Goodrich, Tamassia & Goldwasser).

**Pregunta central:** si lo único que tenemos es una `UnsortedPriorityQueue` de Goodrich (más *una* variable entera auxiliar), ¿es posible implementar

1. un **stack** (LIFO)?
2. una **queue** (FIFO)?

**Respuesta corta: sí, las dos.** La idea es la misma en ambos casos: la cola de prioridad decide el orden de salida **únicamente por la clave**, así que basta con escoger claves que codifiquen *el orden de llegada*.

**Contenido**

0. Preparación — la cola de prioridad que vamos a usar
1. La idea: la clave es un "sello de tiempo"
2. Ejercicio 1 — Stack con una cola de prioridad
3. Ejercicio 2 — Queue con una cola de prioridad
4. Verificación cruzada contra `ArrayStack` y `ArrayQueue`
5. Costo: ¿qué precio pagamos?
6. Preguntas para pensar

---
## 0. Preparación — la cola de prioridad que vamos a usar

Usamos **directamente** `UnsortedPriorityQueue` del repositorio del curso (`goodrich.ch09.unsorted_priority_queue`). Solo necesitamos su interfaz:

| Método | Descripción | Costo |
|---|---|---|
| `add(k, v)` | agrega la entrada con clave `k` y valor `v` | $O(1)$ |
| `min()` | retorna `(k, v)` con la clave **mínima**, sin removerla | $O(n)$ |
| `remove_min()` | retorna y remueve `(k, v)` con la clave mínima | $O(n)$ |
| `len(pq)`, `is_empty()` | tamaño / vacía | $O(1)$ |

Detalle que importa: es una cola **orientada al mínimo**. `remove_min()` siempre entrega la clave *más pequeña*, sin importar cuándo se insertó.

In [9]:
from goodrich.ch09.unsorted_priority_queue import *

In [19]:
PQ = UnsortedPriorityQueue()

In [20]:
PQ.add(2,'Maria')
PQ.add(3.5, 'Carlos')

In [22]:
PQ.min()[1]

'Maria'

In [ ]:
PQ.remove_min(), len(PQ)

In [ ]:
class Empty(Exception):
    pass


class StackPQ():
    def __init__(self):
        # TODO
        pass

    def push(self, e):
        # TODO
        pass

    def top(self):
        # TODO
        pass

    def pop(self):
        # TODO
        pass


In [ ]:
from goodrich.ch09.unsorted_priority_queue import UnsortedPriorityQueue
from goodrich.ch06.array_stack import ArrayStack
from goodrich.ch06.array_queue import ArrayQueue
from goodrich.exceptions import Empty

# --- Prueba rápida: la PQ ignora el orden de llegada ---
pq = UnsortedPriorityQueue()
pq.add(5, 'e'); pq.add(1, 'a'); pq.add(3, 'c')
print(pq.min())                                   # (1, 'a')
print([pq.remove_min() for _ in range(len(pq))])  # [(1, 'a'), (3, 'c'), (5, 'e')]

---
## 1. La idea: la clave es un "sello de tiempo"

Una cola de prioridad no sabe nada de "primero" o "último": solo sabe de **claves**. Pero nosotros elegimos las claves.

Llevamos **una variable entera** `contador` y la usamos como clave de cada elemento que llega:

| Queremos | Claves que asignamos | Por qué funciona |
|---|---|---|
| **Queue** (sale el más *antiguo*) | `0, 1, 2, 3, …` (**crecen**) | el mínimo es el que llegó primero |
| **Stack** (sale el más *reciente*) | `0, -1, -2, -3, …` (**decrecen**) | el mínimo es el que llegó último |

Dos observaciones:

- **Las claves nunca se repiten**, porque el contador avanza en cada inserción. Así se elimina el problema del desempate arbitrario que tiene el ADT (sección 9.1 de la teoría).
- **El valor** `v` de la entrada es el elemento real que el usuario guarda; la clave es solo un detalle interno que el usuario del stack/queue nunca ve.

Trazado del stack: `push('A'), push('B'), push('C'), pop()`

| Paso | Entrada agregada `(clave, valor)` | Contenido de la PQ | `pop()` retorna |
|---|---|---|---|
| `push('A')` | `(0, 'A')` | `{(0,A)}` | |
| `push('B')` | `(-1, 'B')` | `{(0,A), (-1,B)}` | |
| `push('C')` | `(-2, 'C')` | `{(0,A), (-1,B), (-2,C)}` | |
| `pop()` | | mínimo = `(-2,C)` | **`'C'`** (el último en entrar) ✔ |

---
## 2. Ejercicio 1 — Stack con una cola de prioridad

Implementa `PQStack` usando **solo** una `UnsortedPriorityQueue` (`self._pq`) y **una** variable entera (`self._clave`).

- `push(e)`: agrega `e` con una clave **menor que todas las anteriores**.
- `top()`: retorna (sin remover) el elemento más reciente. Lanza `Empty('Stack is empty')` si está vacío.
- `pop()`: remueve y retorna el elemento más reciente. Lanza `Empty('Stack is empty')` si está vacío.
- `len(S)` y `is_empty()`.

*(Completa los métodos marcados con `TODO`; las pruebas de abajo te dirán si quedó bien.)*

In [ ]:
class PQStack:
    '''Stack (LIFO) implementado con una UnsortedPriorityQueue y un entero.'''

    def __init__(self):
        # TODO: la PQ (self._pq) y la única variable auxiliar (self._clave)
        pass

    def __len__(self):
        # TODO
        pass

    def is_empty(self):
        # TODO
        pass

    def push(self, e):
        '''Claves decrecientes: el último en llegar tiene la clave mínima.'''
        # TODO
        pass

    def top(self):
        # TODO
        pass

    def pop(self):
        # TODO
        pass


# --- Pruebas ---
S = PQStack()
for x in 'ABC':
    S.push(x)
print(len(S), S.top())                     # 3 C
print(S.pop(), S.pop(), S.pop())           # C B A
print(S.is_empty())                        # True

S.push(1); S.push(2)
assert S.pop() == 2                        # sigue siendo LIFO tras vaciar y reusar
S.push(3)
assert [S.pop(), S.pop()] == [3, 1]

try:
    S.pop()
except Empty as ex:
    print('Empty:', ex)                    # Empty: Stack is empty

---
## 3. Ejercicio 2 — Queue con una cola de prioridad

Ahora la misma idea, pero con claves **crecientes**. Implementa `PQQueue`:

- `enqueue(e)`: agrega `e` con una clave **mayor que todas las anteriores**.
- `first()`: retorna (sin remover) el elemento más antiguo. Lanza `Empty('Queue is empty')` si está vacía.
- `dequeue()`: remueve y retorna el elemento más antiguo. Lanza `Empty('Queue is empty')` si está vacía.
- `len(Q)` y `is_empty()`.

**Pista:** la única diferencia con `PQStack` es el signo con el que avanza el contador.

In [ ]:
class PQQueue:
    '''Queue (FIFO) implementada con una UnsortedPriorityQueue y un entero.'''

    def __init__(self):
        # TODO: la PQ (self._pq) y la única variable auxiliar (self._clave)
        pass

    def __len__(self):
        # TODO
        pass

    def is_empty(self):
        # TODO
        pass

    def enqueue(self, e):
        '''Claves crecientes: el primero en llegar tiene la clave mínima.'''
        # TODO
        pass

    def first(self):
        # TODO
        pass

    def dequeue(self):
        # TODO
        pass


# --- Pruebas ---
Q = PQQueue()
for x in 'ABC':
    Q.enqueue(x)
print(len(Q), Q.first())                   # 3 A
print(Q.dequeue(), Q.dequeue(), Q.dequeue())   # A B C
print(Q.is_empty())                        # True

Q.enqueue(1); Q.enqueue(2)
assert Q.dequeue() == 1                    # sigue siendo FIFO tras vaciar y reusar
Q.enqueue(3)
assert [Q.dequeue(), Q.dequeue()] == [2, 3]

try:
    Q.dequeue()
except Empty as ex:
    print('Empty:', ex)                    # Empty: Queue is empty

---
## 4. Verificación cruzada contra `ArrayStack` y `ArrayQueue`

Unas pocas pruebas a mano no bastan. Ejecutamos **la misma secuencia aleatoria de operaciones** sobre nuestra versión y sobre la de Goodrich, y exigimos que en todo momento retornen lo mismo (incluyendo las excepciones).

In [ ]:
import random


def resultado(f, *args):
    '''Ejecuta f y retorna su valor, o el marcador 'Empty' si lanza Empty.'''
    try:
        return f(*args)
    except Empty:
        return 'Empty'


def comparar(nuevo, ref, agregar, quitar, ver, pasos=5000, semilla=0):
    rnd = random.Random(semilla)
    for i in range(pasos):
        # más inserciones que remociones, pero con rachas de vaciado
        if rnd.random() < 0.55:
            v = rnd.randint(0, 100)
            getattr(nuevo, agregar)(v)
            getattr(ref, agregar)(v)
        else:
            op = quitar if rnd.random() < 0.7 else ver
            assert resultado(getattr(nuevo, op)) == resultado(getattr(ref, op)), (i, op)
        assert len(nuevo) == len(ref) and nuevo.is_empty() == ref.is_empty()


comparar(PQStack(), ArrayStack(), 'push', 'pop', 'top')
comparar(PQQueue(), ArrayQueue(), 'enqueue', 'dequeue', 'first')
print('OK: PQStack y PQQueue se comportan igual que ArrayStack y ArrayQueue')

---
## 5. Costo: ¿qué precio pagamos?

Que sea **posible** no significa que sea **buena idea**. Con `UnsortedPriorityQueue`:

| Operación | `PQStack` / `PQQueue` | `ArrayStack` / `ArrayQueue` |
|---|---|---|
| `push` / `enqueue` | $O(1)$ | $O(1)$ amortizado |
| `pop` / `dequeue` | **$O(n)$** | $O(1)$ amortizado |
| `top` / `first` | **$O(n)$** | $O(1)$ |

`add` es barato (agrega al final), pero **toda** consulta o remoción del mínimo recorre la lista completa buscando la clave menor. Medimos el tiempo de **un** `pop` después de `n` `push`:

In [ ]:
import time


def tiempo_un_pop(clase_stack, n):
    S = clase_stack()
    for i in range(n):
        S.push(i)
    t0 = time.perf_counter()
    S.pop()
    return time.perf_counter() - t0


print(f"{'n':>7} | {'PQStack (ms)':>13} | {'ArrayStack (ms)':>16}")
for n in (1000, 2000, 4000, 8000, 16000):
    a = tiempo_un_pop(PQStack, n) * 1000
    b = tiempo_un_pop(ArrayStack, n) * 1000
    print(f'{n:>7} | {a:>13.3f} | {b:>16.5f}')

Al duplicar `n`, el tiempo de `PQStack.pop` se duplica (**lineal**), mientras que el de `ArrayStack.pop` no cambia (**constante**).

**Costo de construir *n* elementos y vaciarlos todos:** $n$ inserciones de $O(1)$ más $n$ remociones de $O(n)$ = $O(n^2)$. Es exactamente la misma cuenta que la *selection-sort* de la sección 9.4: usar una PQ desordenada para ordenar es lo mismo que usarla para simular un stack o una queue.

### ¿Y con otras implementaciones de la PQ?

Como `PQStack` y `PQQueue` solo hablan con la interfaz (`add`, `min`, `remove_min`), podemos cambiar la PQ por dentro **sin tocar una línea** del stack o la queue:

| PQ usada por dentro | `push`/`enqueue` | `pop`/`dequeue` |
|---|---|---|
| `UnsortedPriorityQueue` | $O(1)$ | $O(n)$ |
| `SortedPriorityQueue` | $O(n)$ | $O(1)$ |
| `HeapPriorityQueue` | $O(\log n)$ | $O(\log n)$ |

Ninguna alcanza el $O(1)$ de un stack o queue "de verdad": la cola de prioridad **ofrece más de lo que necesitamos** (orden arbitrario por clave) y ese poder extra se paga.

---
## 6. Preguntas para pensar

1. **¿Por qué la variable auxiliar tiene que ser un entero que solo avanza en un sentido?** ¿Qué pasaría en el `PQStack` si, tras un `pop`, retrocedieras el contador (`self._clave += 1`)? Encuentra una secuencia de operaciones que rompa el orden LIFO.
2. **¿Se desborda el contador?** En Python los enteros no tienen límite. ¿Qué pasaría en un lenguaje con enteros de 32 bits después de $2^{31}$ inserciones?
3. **¿Y al revés?** ¿Se puede implementar una cola de prioridad usando **solo stacks** o **solo queues**? ¿Con qué costos? (Piensa en `add` manteniendo una lista ordenada con `ArrayStack` auxiliares.)
4. **Un deque no cabe.** ¿Puedes implementar un *deque* (`add_first`, `add_last`, `delete_first`, `delete_last`) con una sola `UnsortedPriorityQueue`? ¿Qué operación no puedes atender bien y por qué? (Pista: la PQ solo sabe extraer el **mínimo**.)
5. **Claves no numéricas.** Goodrich solo exige un orden total sobre las claves. ¿Serviría usar `time.time()` como clave en lugar de un contador? ¿Qué puede salir mal si dos elementos llegan en el mismo instante?

---
## 7. Ejercicio 3 — Árbol binario guardado en una lista

Implementa `ArbolBinarioLista`: un árbol binario cuyos nodos se guardan en una **lista** de Python (`self.data`), sin objetos nodo ni punteros. La posición de cada nodo en la lista determina su padre y sus hijos (la misma numeración que usa el *heap* de la sección 9.3):

| Nodo en la posición `i` | Posición |
|---|---|
| raíz | `0` |
| hijo izquierdo | `2*i + 1` |
| hijo derecho | `2*i + 2` |
| padre | `(i - 1) // 2` |

Una posición vacía se representa con `None`. Ejemplo: el árbol

```
      A
     / \
    B   C
   /     \
  D       E
```

se guarda como `['A', 'B', 'C', 'D', None, None, 'E']`.

El constructor es solo `self.data = []`: no hay contador ni otras variables, todo se consulta a través de los índices de la lista. Completa los métodos marcados con `TODO`. Las posiciones (`p`) son índices enteros de la lista; lanza `ValueError('Posicion invalida')` si `p` está fuera de rango o apunta a un `None`.

In [ ]:
class ArbolBinarioLista:
    '''Árbol binario almacenado en una lista (raíz en 0, hijos en 2i+1 y 2i+2).'''

    def __init__(self):
        self.data = []   # toda la información vive aquí; todo se consulta por índice

    def __len__(self):
        '''Número de nodos: cuenta los elementos de self.data que no son None.'''
        contador = 0
        for i in self.data:
            if i is not None:
                contador +=1
        return contador
    
    def is_empty(self):
        return len(self)== 0

    def _validar(self, p):
        '''Retorna p si es una posición válida; si no, lanza ValueError.'''
        if p < 0 or p >= len(self.data):
            raise ValueError("Posicion invalida")
        if self.data[p] is None:
            raise ValueError("Posicion invalidaa")
        return p

    def root(self):
        '''Posición de la raíz (0), o None si el árbol está vacío.'''
        if len(self.data)== 0:
            return None 
        if self.data[0] is None:
            return None 
        return 0

    def parent(self, p):
        '''Posición del padre de p, o None si p es la raíz.'''
        self._validar
        if p == 0:
            return None
        return (p-1)//2

    def left(self, p):
        '''Posición del hijo izquierdo de p, o None si no tiene.'''
        self._validar(p)
        izquierdo = 2 * p +1
        if izquierdo >= len(self.data):
            return None
        if self.data[izquierdo] is None:
            return None
        return izquierdo

    def right(self, p):
        '''Posición del hijo derecho de p, o None si no tiene.'''
        self._validar(p)
        derecho = 2 * p + 2
        if derecho >= len(self.data):
            return None
        if self.data[derecho] is None:
            return None
        return derecho
    

    def element(self, p):
        '''Elemento guardado en la posición p.'''
        self._validar(p)
        return self.data[p]
    

    def num_children(self, p):
        self._validar(p)
        contador == 0
        if self.left(p) is not None:
            contador+=1
        if self.right(p) is not None:
            contador+=1
        return contador
    

    def is_leaf(self, p):
        return self.num_children(p)==0
        

    def add_root(self, e):
        '''Crea la raíz con e. Lanza ValueError('Root exists') si ya hay una.'''
        if self.root is not None:
            raise ValueError("Root exists")
        self.data=[e]
        return 0
    

    def add_left(self, p, e):
        '''Agrega e como hijo izquierdo de p (crece la lista si hace falta).
        Lanza ValueError('Left child exists') si ya existe. Retorna la nueva posición.'''
        self._validar(p)
        if self.left(p) is not None:
            raise ValueError ('Left child exists')
        izquierdo = 2 * p +1
        while len(self.data)<= izquierdo:
            self.data.append(None)
        self.data[izquierdo]=e
        return izquierdo 

    def add_right(self, p, e):
        '''Agrega e como hijo derecho de p. Lanza ValueError('Right child exists') si ya existe.'''
        self._validar(p)
        if self.right(p) is not None:
            raise ValueError ('Right child exists')
        derecho = 2 * p + 2
        while len(self.data)<= derecho:
            self.data.append(None)
        self.data[derecho]=e
        return derecho 

        

    def replace(self, p, e):
        '''Reemplaza el elemento en p y retorna el anterior.'''
        self._validar(p)
        anterior = self.data[p]
        self.data[p]=e
        return anterior
        

    def height(self, p=None):
        '''Altura del subárbol en p (por defecto, de la raíz). Una hoja tiene altura 0.'''
        if p is None:
            p=self.root()
        if p is None:
            return -1
        self._validar(p)
        if self.is_leaf(p):
            return 0
        mayor=0
        izquierdo = self.left(p)
        if izquierdo is not None:
            altura_izq= self.height(izquierdo)
            if altura_izq > mayor:
                mayor = altura_izq
        derecho = self.right(p)
        if derecho is not None:
            altura_der=self.height(derecho)
            if altura_der > mayor:
                mayor= altura_der
        return 1 + mayor 
        

    def preorder(self, p=None):
        '''Lista de elementos en preorden (raíz, izquierdo, derecho).'''
        
        pass

    def inorder(self, p=None):
        '''Lista de elementos en inorden (izquierdo, raíz, derecho).'''
        # TODO
        pass

    def postorder(self, p=None):
        '''Lista de elementos en postorden (izquierdo, derecho, raíz).'''
        # TODO
        pass


# --- Pruebas ---
T = ArbolBinarioLista()
assert T.is_empty() and T.root() is None
r = T.add_root('A')
b = T.add_left(r, 'B')
c = T.add_right(r, 'C')
d = T.add_left(b, 'D')
e = T.add_right(c, 'E')
print(T.data)                                  # ['A', 'B', 'C', 'D', None, None, 'E']
assert len(T) == 5 and T.num_children(r) == 2 and T.is_leaf(d)
assert T.parent(e) == c and T.parent(r) is None
assert T.left(c) is None and T.right(c) == e
assert T.height() == 2
print(T.preorder())                             # ['A', 'B', 'D', 'C', 'E']
print(T.inorder())                              # ['D', 'B', 'A', 'C', 'E']
print(T.postorder())                            # ['D', 'B', 'E', 'C', 'A']
assert T.replace(d, 'Z') == 'D' and T.element(d) == 'Z'

for f in (lambda: T.add_root('X'), lambda: T.add_left(r, 'X'), lambda: T.element(5)):
    try:
        f()
    except ValueError as ex:
        print('ValueError:', ex)


AssertionError: 